In [1]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 413.2 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 479.7 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 522.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 517.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 1.3 GB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 461.4 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 511.4 MB/s  0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 491.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Prepare dataset
%cd ~/SageMaker/
! rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/zalando-research/fashionmnist"

!mkdir dataset -p
!unzip -q -o ./dataset.zip -d ./dataset/

/home/ec2-user/SageMaker


In [3]:
# Import pip packages
import os
import struct

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [4]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [5]:
# Upload dataset zip file to S3
inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [28]:
from PIL import Image

def save_sample_images(df, out_dir, num_images=2):
    os.makedirs(out_dir, exist_ok=True)

    pixels = df.iloc[:, 1:].values   # (N, 784), 아직 0~255 정수값 그대로
    labels = df.iloc[:, 0].values

    for i in range(num_images):
        # (784,) -> (28, 28)로 reshape, 타입은 uint8로 변환
        img_array = pixels[i].reshape(28, 28).astype(np.uint8)

        # numpy 배열 -> PIL 이미지 객체 (mode='L'은 흑백을 의미)
        img = Image.fromarray(img_array, mode='L')

        # 라벨을 파일명에 포함해서 저장
        img.save(os.path.join(out_dir, f"sample_{i}_label{labels[i]}.png"))

In [32]:
test_df = pd.read_csv(f'./dataset/fashion-mnist_test.csv')

save_sample_images(test_df, out_dir=f'./assets', num_images=10)

In [21]:
# Test local

from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",
    framework_version="2.6",
    py_version="py312",
    instance_type="local",
    instance_count=1,
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 32,
        'batch_size': 2,
        'lr': 0.001,
        # 'backend': 'gloo'  # distributed
    },
    metric_definitions    = [
        {"Name": "train:loss", "Regex": r"Train Loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"Train Acc: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"Test Loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"Test Acc: ([0-9\.]+)"},
    ],
)

estimator.fit({
    "train": TrainingInput("s3://{}/{}/dataset.zip".format(bucket, prefix)),  # /opt/ml/input/data/train
})

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2026-07-03-03-41-18-098
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliv

 Container 6wr4egf92e-algo-1-ix6yn  Creating
 Container 6wr4egf92e-algo-1-ix6yn  Created
Attaching to 6wr4egf92e-algo-1-ix6yn
6wr4egf92e-algo-1-ix6yn  | 2026-07-03 03:41:20,895 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
6wr4egf92e-algo-1-ix6yn  | 2026-07-03 03:41:20,896 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
6wr4egf92e-algo-1-ix6yn  | 2026-07-03 03:41:20,896 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
6wr4egf92e-algo-1-ix6yn  | 2026-07-03 03:41:20,902 sagemaker-training-toolkit INFO     instance_groups entry not present in resource_config
6wr4egf92e-algo-1-ix6yn  | 2026-07-03 03:41:20,904 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
6wr4egf92e-algo-1-ix6yn  | 2026-07-03 03:41:20,905 sagemaker_pytorch_container.training INFO     Invoking user training script.
6wr4egf92e-algo-1-ix6yn  | 2026-07-03 03:41:21,878

In [22]:
estimator = PyTorch(
    entry_point="train.py",
    framework_version="2.6",
    py_version="py312",
    instance_type="ml.m6i.xlarge",
    instance_count=1,
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    checkpoint_s3_uri="s3://{}/{}/checkpoint/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 20,
        'batch_size': 64,
        'lr': 0.001,
        # 'backend': 'gloo'  # distributed
    },
    metric_definitions    = [
        {"Name": "train:loss", "Regex": r"Train Loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"Train Acc: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"Test Loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"Test Acc: ([0-9\.]+)"},
    ],
)

estimator.fit({
    "train": TrainingInput("s3://{}/{}/dataset.zip".format(bucket, prefix)),  # /opt/ml/input/data/train
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2026-07-03-03-42-11-705


2026-07-03 03:42:14 Starting - Starting the training job...
2026-07-03 03:42:29 Starting - Preparing the instances for training...
2026-07-03 03:43:08 Downloading - Downloading the training image.....
2026-07-03 03:43:58 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-07-03 03:44:01,492 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-07-03 03:44:01,493 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-07-03 03:44:01,493 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-07-03 03:44:01,501 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-07-03 03:44:01,502 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-07-03 03:44:02,552 sagemaker-training-toolkit 

In [23]:
# Create new model + Endpoint Configuration
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model(
  name = f'fashion-model-{int(time.time())}',
)
model_name = model.name
model._create_sagemaker_model(instance_type="ml.g5.xlarge")

new_config = f'fashendpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.g5.xlarge"
    }]
)

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-200148130345/sagemaker/output/pytorch-training-2026-07-03-03-42-11-705/output/model.tar.gz), script artifact (s3://sagemaker-us-east-1-200148130345/pytorch-training-2026-07-03-03-42-11-705/source/sourcedir.tar.gz), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-200148130345/fashion-model-1783051395/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: fashion-model-1783051395


{'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint-config/fashendpoint-configuration-1783051397',
 'ResponseMetadata': {'RequestId': '1d0c1e0f-311a-4115-8d32-a0d8d323ca33',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '1d0c1e0f-311a-4115-8d32-a0d8d323ca33',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '118',
   'date': 'Fri, 03 Jul 2026 04:03:18 GMT'},
  'RetryAttempts': 0}}

In [24]:
# Create/Update Endpoint
# sm.create_endpoint(
#     EndpointName = "fashion-endpoint",
#     EndpointConfigName = new_config
# )

sm.update_endpoint(
    EndpointName = "fashion-endpoint",
    EndpointConfigName = new_config
)

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint/fashion-endpoint',
 'ResponseMetadata': {'RequestId': '09b22624-ebbb-4c84-80a9-3741cc0d0759',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '09b22624-ebbb-4c84-80a9-3741cc0d0759',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '84',
   'date': 'Fri, 03 Jul 2026 04:03:27 GMT'},
  'RetryAttempts': 0}}

In [18]:
CLASSES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
CLASSES

['T-shirt',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [36]:
# Inference test
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'fashion-endpoint'
image_path = "tshirt.png"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 0, 'confidence': 1.0}
T-shirt


In [40]:
# Inference test
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'fashion-endpoint'
image_path = "trouser.png"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 1, 'confidence': 1.0}
Trouser
